In [7]:
# ============================================================================
# 0. ENVIRONMENT SETUP
# ============================================================================
import numpy as np
import pandas as pd
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, LSTM, Dropout, Bidirectional, Input,
    LeakyReLU, BatchNormalization, RepeatVector, TimeDistributed
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU Enabled: {len(gpus)}")
else:
    print("⚠️ Running on CPU (training will be slower)")


✅ GPU Enabled: 1


In [8]:

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
# Paths — adjust if your project root is different
# Paths — adjust if your project root is different
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined in Jupyter/Colab
    PROJECT_ROOT = os.getcwd()

# If running from the project root directly:
if not os.path.exists(os.path.join(PROJECT_ROOT, 'models')):
    # Try one level up if inside a subdirectory (e.g. GAN_MODEL)
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
    if not os.path.exists(os.path.join(PROJECT_ROOT, 'models')):
        # Fallback to current working directory if all else fails
        PROJECT_ROOT = os.getcwd()

# ----------------------------------------------------------------------------
# COLAB HELPER: Asset Verification & Google Drive Mount
# ----------------------------------------------------------------------------
def ensure_colab_assets():
    """Checks for files. If missing, suggests Drive mount or Drag-and-Drop."""
    import sys
    if 'google.colab' not in sys.modules:
        return

    # Expected artifacts relative to PROJECT_ROOT
    required = [
        os.path.join('models', 'saved_models', 'gan_generator_ohlcv.h5'),
        os.path.join('models', 'saved_models', 'lstm_model.h5'),
        os.path.join('models', 'scalers', 'lstm_model_scaler.pkl'),
        os.path.join('data', 'raw', 'ogdc_data_updated.csv')
    ]
    
    missing = []
    for rel_path in required:
        full_path = os.path.join(PROJECT_ROOT, rel_path)
        if not os.path.exists(full_path):
            missing.append(rel_path)
    
    if not missing:
        print("✅ All assets found.")
        return

    print("\n" + "!"*80)
    print(f"⚠️ MISSING {len(missing)} FILES IN COLAB RUNTIME:")
    for rel in missing:
        print(f"   - {rel}")
    print("!"*80)
    print("\nOPTION 1: GOOGLE DRIVE (Recommended)")
    print("   1. Upload your 'models' and 'data' folders to your Google Drive root.")
    print("   2. Run the cell below to mount Drive.")
    print("   3. The script will link '/content/drive/MyDrive/...' to the expected paths.")
    
    print("\nOPTION 2: DRAG & DROP")
    print("   1. Open the file explorer in the left sidebar.")
    print("   2. Drag your local 'models' and 'data' folders into the runtime.")
    print("!"*80 + "\n")

    # Attempt to mount drive automatically
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            print("Mounting Google Drive...")
            drive.mount('/content/drive')
        
        # Check if files exist in Drive and symlink them
        drive_root = '/content/drive/MyDrive/GAN PSX Marketforecasting'
        # If user just dumped them in root:
        if not os.path.exists(drive_root):
            drive_root = '/content/drive/MyDrive'
            
        print(f"Checking Drive path: {drive_root}")
        found_in_drive = False
        for rel in missing:
            drive_path = os.path.join(drive_root, rel)
            if os.path.exists(drive_path):
                # Create local directory structure
                local_dest = os.path.join(PROJECT_ROOT, rel)
                os.makedirs(os.path.dirname(local_dest), exist_ok=True)
                # Copy or Symlink
                if not os.path.exists(local_dest):
                    import shutil
                    shutil.copy(drive_path, local_dest)
                    print(f"✅ Restored from Drive: {rel}")
                    found_in_drive = True
        
        if found_in_drive:
             print("Assets restored from Google Drive! proceeding...")
             return

    except Exception as e:
        print(f"Drive mount verification failed/skipped: {e}")

    print("\n❌ STILL MISSING FILES. Please upload them manually to continue.")
    # Stop execution if files are strictly required
    # raise FileNotFoundError("Missing critical training files.")

# Run the check before defining strict paths
ensure_colab_assets()
# ----------------------------------------------------------------------------

GAN_WEIGHTS_PATH   = os.path.join(PROJECT_ROOT, 'models', 'saved_models', 'gan_generator_ohlcv.h5')
LSTM_WEIGHTS_PATH  = os.path.join(PROJECT_ROOT, 'models', 'saved_models', 'lstm_model.h5')
CALIBRATED_PATH    = os.path.join(PROJECT_ROOT, 'models', 'saved_models', 'lstm_model_calibrated.h5')
SCALER_PATH        = os.path.join(PROJECT_ROOT, 'models', 'scalers', 'lstm_model_scaler.pkl')
DATA_PATH          = os.path.join(PROJECT_ROOT, 'data', 'raw', 'ogdc_data_updated.csv')

# Hyperparameters
LOOKBACK            = 60       # Sequence length for LSTM
LATENT_DIM          = 100      # GAN noise dimension
GAN_SEQ_LEN         = 30       # GAN output sequence length
GAN_NUM_FEATURES    = 5        # OHLCV
CLOSE_COL_IDX       = 3        # Column index for Close in GAN output

N_SYNTHETIC_DAYS    = 5000     # How many synthetic closing prices to generate
STAGE_A_EPOCHS      = 30       # Pre-training epochs on synthetic data
STAGE_B_EPOCHS      = 50       # Fine-tuning epochs on real data
BATCH_SIZE          = 32
LEARNING_RATE_A     = 0.001    # Stage A learning rate (broad generalization)
LEARNING_RATE_B     = 0.0003   # Stage B learning rate (fine-tuning — lower)

print("\n" + "="*70)
print("  NEXUS AI — GAN→LSTM Augmented Training")
print("="*70)
print(f"  GAN Weights:      {GAN_WEIGHTS_PATH}")
print(f"  LSTM Weights:     {LSTM_WEIGHTS_PATH}")
print(f"  Output (calib):   {CALIBRATED_PATH}")
print(f"  Real Data:        {DATA_PATH}")
print(f"  Synthetic Days:   {N_SYNTHETIC_DAYS}")
print(f"  Stage A Epochs:   {STAGE_A_EPOCHS}")
print(f"  Stage B Epochs:   {STAGE_B_EPOCHS}")
print("="*70 + "\n")


Mounted at /content/drive
Checking Drive path: /content/drive/MyDrive
✅ Restored from Drive: models/saved_models/gan_generator_ohlcv.h5
✅ Restored from Drive: models/saved_models/lstm_model.h5
✅ Restored from Drive: models/scalers/lstm_model_scaler.pkl
✅ Restored from Drive: data/raw/ogdc_data_updated.csv
Assets restored from Google Drive! proceeding...

  NEXUS AI — GAN→LSTM Augmented Training
  GAN Weights:      /content/models/saved_models/gan_generator_ohlcv.h5
  LSTM Weights:     /content/models/saved_models/lstm_model.h5
  Output (calib):   /content/models/saved_models/lstm_model_calibrated.h5
  Real Data:        /content/data/raw/ogdc_data_updated.csv
  Synthetic Days:   5000
  Stage A Epochs:   30
  Stage B Epochs:   50

